# IFRS 9 Expected Credit Loss (ECL) Model
## Notebook 1 — Loan Portfolio Overview

This notebook provides an overview of the synthetic loan portfolio used for ECL modeling:
- Portfolio composition by product type and sector
- Credit quality distribution
- Days Past Due (DPD) analysis
- IFRS 9 Stage assignment

**Reference:** IFRS 9 Financial Instruments (IASB, 2014) | Bank of Thailand IFRS 9 Implementation Guidelines

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import generate_loan_portfolio, assign_stage, stage_summary

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_style('whitegrid')

df_raw = generate_loan_portfolio(n_loans=5000, random_state=42)
df = assign_stage(df_raw)

print(f'Portfolio: {len(df):,} loans | Total EAD: THB {df["outstanding_balance"].sum()/1e9:.2f}B')
df.head()

## 1. Portfolio Composition

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

prod_bal = df.groupby('product_type')['outstanding_balance'].sum() / 1e6
prod_bal.sort_values().plot(kind='barh', ax=axes[0], color='#2980b9', alpha=0.85)
axes[0].set_title('Outstanding Balance by Product Type (THB M)')
axes[0].set_xlabel('THB Million')

sec_bal = df.groupby('sector')['outstanding_balance'].sum() / 1e6
sec_bal.sort_values().plot(kind='barh', ax=axes[1], color='#e67e22', alpha=0.85)
axes[1].set_title('Outstanding Balance by Sector (THB M)')
axes[1].set_xlabel('THB Million')

plt.suptitle('Portfolio Composition', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/01_portfolio_composition.png', bbox_inches='tight')
plt.show()

## 2. Credit Quality Distribution

In [ ]:
rating_labels = {1:'AAA',2:'AA',3:'A',4:'BBB',5:'BB',6:'B',7:'CCC',8:'Default'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in [
    (axes[0], 'credit_rating_orig', 'Rating at Origination'),
    (axes[1], 'credit_rating_curr', 'Rating at Reporting Date'),
]:
    counts = df[col].value_counts().sort_index()
    colors = ['#27ae60','#2ecc71','#f1c40f','#e67e22','#e74c3c','#c0392b','#8e44ad','#2c3e50']
    ax.bar([rating_labels.get(r, str(r)) for r in counts.index],
           counts.values, color=colors[:len(counts)], alpha=0.85)
    ax.set_title(title)
    ax.set_ylabel('Number of Loans')
    ax.set_xlabel('Rating Grade')

plt.suptitle('Credit Rating Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/01_rating_distribution.png', bbox_inches='tight')
plt.show()

# Rating migration summary
df['rating_change'] = df['credit_rating_curr'] - df['credit_rating_orig']
print('Rating Migration Summary:')
print(df['rating_change'].value_counts().sort_index())

## 3. Days Past Due (DPD) Distribution

In [ ]:
dpd_bins = [0, 1, 30, 60, 90, 180, 366]
dpd_labels = ['Current (0)', '1–29 DPD', '30–59 DPD', '60–89 DPD', '90–179 DPD', '180+ DPD']
df['dpd_bucket'] = pd.cut(df['days_past_due'], bins=dpd_bins, labels=dpd_labels, right=False)

dpd_count = df.groupby('dpd_bucket', observed=True)['loan_id'].count()
dpd_balance = df.groupby('dpd_bucket', observed=True)['outstanding_balance'].sum() / 1e6

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dpd_colors = ['#27ae60','#f1c40f','#e67e22','#e74c3c','#c0392b','#8e44ad']

dpd_count.plot(kind='bar', ax=axes[0], color=dpd_colors, alpha=0.85, edgecolor='white')
axes[0].set_title('Loan Count by DPD Bucket')
axes[0].set_ylabel('Number of Loans')
axes[0].tick_params(axis='x', rotation=30)

dpd_balance.plot(kind='bar', ax=axes[1], color=dpd_colors, alpha=0.85, edgecolor='white')
axes[1].set_title('Outstanding Balance by DPD Bucket (THB M)')
axes[1].set_ylabel('THB Million')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Days Past Due Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/01_dpd_distribution.png', bbox_inches='tight')
plt.show()

## 4. IFRS 9 Stage Assignment

In [ ]:
summary = stage_summary(df)
print('=== IFRS 9 Stage Summary ===')
print(summary[['n_loans','pct_count','outstanding_balance','pct_balance','avg_dpd','stage_label']])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
stage_colors = ['#2196F3', '#FF9800', '#F44336']
stage_labels = [f'Stage {s}' for s in summary.index]

axes[0].pie(summary['n_loans'], labels=stage_labels, colors=stage_colors,
            autopct='%1.1f%%', startangle=90)
axes[0].set_title('Loan Count by Stage')

axes[1].pie(summary['outstanding_balance'], labels=stage_labels, colors=stage_colors,
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Outstanding Balance by Stage')

plt.suptitle('IFRS 9 Stage Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/01_stage_distribution.png', bbox_inches='tight')
plt.show()